# MILK10k — Step 4b: calibrated mixing, class-aware policies, bootstrap CIs
Follow-up to notebook 04. Three changes:
1. **Per-pattern temperature scaling** (cross-fitted). Mixing CM and CDM scores is only fair if both are on the same probability scale; uncalibrated mixing can depress AUC for any policy.
2. **New policies**: `class_benefit` (expected per-class dermoscopy gain, learned on other folds) and `hybrid` (uncertainty + class benefit), aimed at *confident* errors that uncertainty misses (e.g. SCCKA).
3. **Actionable decisions at a fixed specificity** (threshold set on other folds) and **bootstrap 95% CIs** for the headline rate.

In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score, roc_auc_score, recall_score

def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None
NPZ = find_file("oof_probs_fusion.npz"); FOLDS_CSV = find_file("milk10k_lesions_folds.csv")
assert NPZ and FOLDS_CSV, "attach notebook-o3 and notebook-o2 outputs"
OUT = "/kaggle/working/stage4b"; os.makedirs(OUT, exist_ok=True)
les = pd.read_csv(FOLDS_CSV)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; act_idx = [CLASSES.index(c) for c in ACTIONABLE]
y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y)
y_act = np.isin(y, act_idx).astype(int)
npz = np.load(NPZ); eps = 1e-9
RAW = {p: npz[f"late_drop|{p}"].astype(float) for p in ["CM", "CDM"]}

## 1. Cross-fitted temperature scaling (one temperature per pattern, fitted on the other 4 folds)

In [ ]:
def nll(P, yy): return -np.log(P[np.arange(len(yy)), yy] + eps).mean()
def temp(P, T):
    L = np.log(P + eps) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def ece(P, yy, bins=15):
    conf, acc = P.max(1), (P.argmax(1) == yy); e = 0
    for lo_, hi_ in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo_) & (conf <= hi_)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

CAL, temps = {}, {}
for p, P in RAW.items():
    out = np.zeros_like(P)
    for f in range(5):
        tr, te = fold != f, fold == f
        T = minimize_scalar(lambda t: nll(temp(P[tr], t), y[tr]), bounds=(0.2, 5), method="bounded").x
        temps[(p, f)] = T; out[te] = temp(P[te], T)
    CAL[p] = out
print(pd.DataFrame({p: {"T (mean over folds)": np.mean([temps[(p, f)] for f in range(5)]),
                        "NLL raw": nll(RAW[p], y), "NLL cal": nll(CAL[p], y),
                        "ECE raw": ece(RAW[p], y), "ECE cal": ece(CAL[p], y)} for p in RAW}).round(3))
P_CM, P_CDM = CAL["CM"], CAL["CDM"]

## 2. Actionable threshold at fixed specificity (set on other folds, per pattern)

In [ ]:
TARGET_SPEC = 0.70
def act_score(P): return P[:, act_idx].sum(1)
thr = {}
for p, P in CAL.items():
    for f in range(5):
        tr = (fold != f) & (y_act == 0)
        thr[(p, f)] = np.quantile(act_score(P)[tr], TARGET_SPEC)     # spec = TARGET_SPEC on training folds
thr_vec = {p: np.array([thr[(p, f)] for f in fold]) for p in CAL}

def metrics(P, req):
    pred = P.argmax(1); s = act_score(P)
    t = np.where(req, thr_vec["CDM"], thr_vec["CM"])              # each lesion uses its own pattern's threshold
    flag = s >= t
    return {"macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
            "act_auc": roc_auc_score(y_act, s),
            "act_sens@spec": (flag & (y_act == 1)).sum() / y_act.sum(),
            "act_spec": (~flag & (y_act == 0)).sum() / (y_act == 0).sum(),
            "MEL_recall": recall_score(y == 7, pred == 7), "SCCKA_recall": recall_score(y == 9, pred == 9)}
none, allr = np.zeros(n, bool), np.ones(n, bool)
lo, hi = metrics(P_CM, none), metrics(P_CDM, allr)
print(pd.DataFrame({"CM only": lo, "CDM for all": hi}).round(3))

## 3. Policies (pre-dermoscopy information only, except oracle)

In [ ]:
def entropy(P): return -(P * np.log(P + eps)).sum(1)
top2 = np.sort(P_CM, 1)[:, -2:]
# per-class expected benefit of dermoscopy, estimated on the other folds:
# gain_k = mean over training lesions of class k of [log P_CDM(y) - log P_CM(y)]
gain = np.log(P_CDM[np.arange(n), y] + eps) - np.log(P_CM[np.arange(n), y] + eps)
class_benefit = np.zeros(n)
for f in range(5):
    tr, te = fold != f, fold == f
    g_k = np.array([gain[tr & (y == k)].mean() if (tr & (y == k)).any() else 0 for k in range(K)])
    class_benefit[te] = P_CM[te] @ g_k                         # expected gain under the CM belief
def z(v): return (v - v.mean()) / (v.std() + eps)
scores = {"random": np.random.RandomState(0).rand(n),
          "entropy": entropy(P_CM), "least_conf": 1 - P_CM.max(1), "small_margin": -(top2[:, 1] - top2[:, 0]),
          "class_benefit": class_benefit,
          "hybrid": z(entropy(P_CM)) + z(class_benefit),
          "oracle": gain}
print("Per-class mean dermoscopy gain (all folds, for reference):")
print(pd.Series([gain[y == k].mean() for k in range(K)], index=CLASSES).round(3).sort_values(ascending=False))

## 4. Curves + headline (rate needed for 95% of the gain)

In [ ]:
RATES = np.round(np.arange(0, 1.0001, 0.05), 2)
def req_mask(score, q, idx=None):
    idx = np.arange(n) if idx is None else idx
    r = np.zeros(n, bool); k = int(round(q * len(idx)))
    if k: r[idx[np.argsort(-score[idx])[:k]]] = True
    return r
rows = []
for name, s in scores.items():
    for q in RATES:
        r = req_mask(s, q); rows.append(dict(policy=name, rate=q, **metrics(np.where(r[:, None], P_CDM, P_CM), r)))
curves = pd.DataFrame(rows); curves.to_csv(f"{OUT}/curves_calibrated.csv", index=False)
METRICS = ["macro_f1", "act_auc", "act_sens@spec"]
def rate_needed(d, m, frac=0.95):
    tgt = lo[m] + frac * (hi[m] - lo[m]); ok = d[d[m] >= tgt]
    return ok["rate"].min() if len(ok) else np.nan
head = pd.DataFrame({m: {p: rate_needed(d, m) for p, d in curves.groupby("policy")} for m in METRICS})
head.to_csv(f"{OUT}/headline_calibrated.csv"); head.sort_values("macro_f1")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
sty = {"random": ("gray", "--", 1.3), "oracle": ("black", ":", 1.3), "hybrid": ("C3", "-", 2.4), "class_benefit": ("C4", "-", 1.8)}
for m, a in zip(METRICS, ax):
    for p, d in curves.groupby("policy"):
        col, ls, lw = sty.get(p, (None, "-", 1.2)); a.plot(d["rate"] * 100, d[m], label=p, color=col, ls=ls, lw=lw)
    a.axhline(hi[m], color="C2", lw=0.8); a.set_title(m); a.set_xlabel("% of lesions sent for dermoscopy")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{OUT}/fig_curves_calibrated.png", dpi=200); plt.show()

## 5. Bootstrap 95% CI for the headline (macro F1, lesions resampled with replacement)

In [ ]:
def boot_rate(score, B=300, seed=0):
    rng = np.random.RandomState(seed); out = []
    for _ in range(B):
        idx = rng.randint(0, n, n); yy = y[idx]; vals = []
        for q in RATES:
            k = int(round(q * n)); r = np.zeros(n, bool)
            if k: r[np.argsort(-score[idx])[:k]] = True
            P = np.where(r[:, None], P_CDM[idx], P_CM[idx])
            vals.append(f1_score(yy, P.argmax(1), average="macro", labels=range(K), zero_division=0))
        v = np.array(vals); tgt = v[0] + 0.95 * (v[-1] - v[0]); out.append(RATES[np.argmax(v >= tgt)])
    return np.percentile(out, [2.5, 50, 97.5])
ci = pd.DataFrame({p: boot_rate(scores[p]) for p in ["random", "entropy", "least_conf", "class_benefit", "hybrid"]},
                  index=["2.5%", "median", "97.5%"]).T
ci.to_csv(f"{OUT}/headline_bootstrap_ci.csv"); ci

## 6. Confident errors: CM mistakes that uncertainty policies would NOT send (top-30% rule)

In [ ]:
r_unc = req_mask(scores["least_conf"], 0.30); r_hyb = req_mask(scores["hybrid"], 0.30)
wrong_CM = P_CM.argmax(1) != y; fixed_by_D = P_CDM.argmax(1) == y
tab = pd.DataFrame({
    "n": [(y == k).sum() for k in range(K)],
    "CM wrong": [(wrong_CM & (y == k)).sum() for k in range(K)],
    "...and dermoscopy fixes": [(wrong_CM & fixed_by_D & (y == k)).sum() for k in range(K)],
    "fixable, missed by least_conf@30%": [(wrong_CM & fixed_by_D & ~r_unc & (y == k)).sum() for k in range(K)],
    "fixable, missed by hybrid@30%": [(wrong_CM & fixed_by_D & ~r_hyb & (y == k)).sum() for k in range(K)]},
    index=CLASSES)
tab.loc["TOTAL"] = tab.sum(); tab.to_csv(f"{OUT}/confident_errors.csv"); tab